In [1]:
import pandas as pd
import numpy as np
import os

from pathlib import Path
from dotenv import load_dotenv
from google import genai

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
env_file = r"C:\Packaging_AI_Control_Tower\.env"

load_dotenv(env_file, override=True)

api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    print("Gemini API key loaded successfully")
else:
    print("Gemini API key NOT found")

Gemini API key loaded successfully


In [3]:
client = genai.Client(
    api_key=api_key
)

print("Gemini client created successfully")

Gemini client created successfully


In [4]:
features_path = r"C:\Packaging_AI_Control_Tower\Data\processed\features.csv"
anomaly_path = r"C:\Packaging_AI_Control_Tower\Data\processed\anomaly_results.csv"
risk_path = r"C:\Packaging_AI_Control_Tower\Data\processed\risk_results.csv"

features_df = pd.read_csv(features_path)
anomaly_df = pd.read_csv(anomaly_path)
risk_df = pd.read_csv(risk_path)

print("Features:", features_df.shape)
print("Anomaly:", anomaly_df.shape)
print("Risk:", risk_df.shape)

Features: (23376, 170)
Anomaly: (23376, 174)
Risk: (4670, 177)


In [5]:
def collect_rca_evidence(equipment_id):

    # Machine operational data
    machine_data = features_df[
        features_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    # Anomaly results
    anomaly_data = anomaly_df[
        anomaly_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    # Risk results
    risk_data = risk_df[
        risk_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    return {
        "machine_data": machine_data,
        "anomaly_data": anomaly_data,
        "risk_data": risk_data
    }

In [6]:
print(features_df["equipment_ID"].unique()[:20])

<ArrowStringArray>
['s_1', 's_2', 's_3', 's_4', 's_5']
Length: 5, dtype: str


In [11]:
evidence = collect_rca_evidence(
    equipment_id
)

print("Machine records:", len(evidence["machine_data"]))
print("Anomaly records:", len(evidence["anomaly_data"]))
print("Risk records:", len(evidence["risk_data"]))

Machine records: 0
Anomaly records: 0
Risk records: 0


In [12]:
machine_text = evidence["machine_data"].tail(10).to_string(
    index=False
)

anomaly_text = evidence["anomaly_data"].tail(10).to_string(
    index=False
)

risk_text = evidence["risk_data"].tail(10).to_string(
    index=False
)

In [13]:
machine_data = evidence["machine_data"]

avg_production = machine_data["%production"].mean()
avg_downtime = machine_data["%downtime"].mean()
avg_idle = machine_data["%idle"].mean()
avg_performance_loss = machine_data["%performance_loss"].mean()
avg_health = machine_data["health_score"].mean()

print("Average Production:", avg_production)
print("Average Downtime:", avg_downtime)
print("Average Idle:", avg_idle)
print("Average Performance Loss:", avg_performance_loss)
print("Average Health Score:", avg_health)

Average Production: nan
Average Downtime: nan
Average Idle: nan
Average Performance Loss: nan
Average Health Score: nan


In [14]:
anomaly_data = evidence["anomaly_data"]

if "is_anomaly" in anomaly_data.columns:
    anomaly_count = anomaly_data["is_anomaly"].sum()
else:
    anomaly_count = 0

print("Anomalous records:", anomaly_count)

Anomalous records: 0


In [15]:
if anomaly_count > 0:
    anomaly_status = "ANOMALY DETECTED"
else:
    anomaly_status = "NO ANOMALY DETECTED"

print(anomaly_status)

NO ANOMALY DETECTED


In [16]:
risk_data = evidence["risk_data"]

if "risk_prediction" in risk_data.columns:
    risk_count = risk_data["risk_prediction"].sum()
else:
    risk_count = 0

print("High-risk predictions:", risk_count)

High-risk predictions: 0


In [17]:
print("Number of machines:",
      features_df["equipment_ID"].nunique())

print("\nActual machine IDs:")
print(features_df["equipment_ID"].unique()[:30])

Number of machines: 5

Actual machine IDs:
<ArrowStringArray>
['s_1', 's_2', 's_3', 's_4', 's_5']
Length: 5, dtype: str


In [18]:
equipment_id = "M001"

print("Records found:",
      len(features_df[
          features_df["equipment_ID"].astype(str) == str(equipment_id)
      ]))

Records found: 0


In [19]:
equipment_id = str(
    features_df["equipment_ID"].dropna().iloc[0]
)

print("Selected machine:", equipment_id)

Selected machine: s_1


In [20]:
evidence = collect_rca_evidence(equipment_id)

print("Machine records:",
      len(evidence["machine_data"]))

print("Anomaly records:",
      len(evidence["anomaly_data"]))

print("Risk records:",
      len(evidence["risk_data"]))

Machine records: 8973
Anomaly records: 8973
Risk records: 1294


In [21]:
machine_data = evidence["machine_data"]

avg_production = machine_data["%production"].mean()
avg_downtime = machine_data["%downtime"].mean()
avg_idle = machine_data["%idle"].mean()
avg_performance_loss = machine_data["%performance_loss"].mean()
avg_health = machine_data["health_score"].mean()

print("Average Production:", avg_production)
print("Average Downtime:", avg_downtime)
print("Average Idle:", avg_idle)
print("Average Performance Loss:", avg_performance_loss)
print("Average Health Score:", avg_health)

Average Production: 0.6325478319986719
Average Downtime: 0.1595151755282708
Average Idle: 0.03571024783004246
Average Performance Loss: 0.11156098648563229
Average Health Score: 99.69321359015605


In [22]:
anomaly_data = evidence["anomaly_data"]

if len(anomaly_data) == 0:
    anomaly_status = "NO DATA AVAILABLE"
elif "is_anomaly" in anomaly_data.columns:
    anomaly_count = int(anomaly_data["is_anomaly"].sum())

    if anomaly_count > 0:
        anomaly_status = "ANOMALY DETECTED"
    else:
        anomaly_status = "NO ANOMALY DETECTED"
else:
    anomaly_status = "ANOMALY DATA UNAVAILABLE"

print("Anomaly Status:", anomaly_status)

Anomaly Status: ANOMALY DETECTED


In [23]:
risk_data = evidence["risk_data"]

if len(risk_data) == 0:
    risk_status = "NO DATA AVAILABLE"
elif "risk_prediction" in risk_data.columns:
    risk_count = int(risk_data["risk_prediction"].sum())

    if risk_count > 0:
        risk_status = "RISK DETECTED"
    else:
        risk_status = "NO HIGH-RISK PREDICTION"
else:
    risk_status = "RISK DATA UNAVAILABLE"

print("Risk Status:", risk_status)

Risk Status: RISK DETECTED


In [24]:
# Find machine IDs available in all three datasets

feature_machines = set(
    features_df["equipment_ID"].dropna().astype(str).unique()
)

anomaly_machines = set(
    anomaly_df["equipment_ID"].dropna().astype(str).unique()
)

risk_machines = set(
    risk_df["equipment_ID"].dropna().astype(str).unique()
)

common_machines = (
    feature_machines
    & anomaly_machines
    & risk_machines
)

print("Common machines:", len(common_machines))
print("Sample machine IDs:", list(common_machines)[:20])

Common machines: 5
Sample machine IDs: ['s_3', 's_4', 's_5', 's_1', 's_2']


In [25]:
equipment_id = sorted(common_machines)[0]

print("Selected Machine:", equipment_id)

Selected Machine: s_1


In [26]:
def collect_rca_evidence(equipment_id):

    machine_data = features_df[
        features_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    anomaly_data = anomaly_df[
        anomaly_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    risk_data = risk_df[
        risk_df["equipment_ID"].astype(str) == str(equipment_id)
    ].copy()

    return {
        "machine_data": machine_data,
        "anomaly_data": anomaly_data,
        "risk_data": risk_data
    }


evidence = collect_rca_evidence(equipment_id)

print("Machine records:", len(evidence["machine_data"]))
print("Anomaly records:", len(evidence["anomaly_data"]))
print("Risk records:", len(evidence["risk_data"]))

Machine records: 8973
Anomaly records: 8973
Risk records: 1294


In [27]:
machine_data = evidence["machine_data"]

print(machine_data.shape)
print(machine_data.head())

(8973, 170)
        interval_start equipment_ID  count_sum  A_028  A_029  A_024  A_045  \
0  2020-01-01 14:00:00          s_1          4    0.0      0      0    NaN   
1  2020-01-01 15:00:00          s_1          2    0.0      0      0    NaN   
2  2020-01-01 17:00:00          s_1          1    0.0      0      0    NaN   
3  2020-01-01 18:00:00          s_1          4    0.0      0      0    NaN   
4  2020-01-01 19:00:00          s_1          1    0.0      0      0    NaN   

   A_001  A_058  A_064  ...  A_053  A_054  A_060  A_061  \
0    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
1    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
2    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
3    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   
4    0.0    NaN    0.0  ...    NaN    NaN    NaN    NaN   

   production_rolling_6h  downtime_rolling_6h  production_change  \
0               0.861729             0.052261           0.000000   
1               0.866313    

In [28]:
avg_production = machine_data["%production"].mean()
avg_downtime = machine_data["%downtime"].mean()
avg_idle = machine_data["%idle"].mean()
avg_performance_loss = machine_data["%performance_loss"].mean()
avg_health = machine_data["health_score"].mean()

print("Average Production:", avg_production)
print("Average Downtime:", avg_downtime)
print("Average Idle:", avg_idle)
print("Average Performance Loss:", avg_performance_loss)
print("Average Health Score:", avg_health)

Average Production: 0.6325478319986719
Average Downtime: 0.1595151755282708
Average Idle: 0.03571024783004246
Average Performance Loss: 0.11156098648563229
Average Health Score: 99.69321359015605


In [29]:
production_pct = avg_production * 100
downtime_pct = avg_downtime * 100
idle_pct = avg_idle * 100
performance_loss_pct = avg_performance_loss * 100

print(f"Average Production: {production_pct:.2f}%")
print(f"Average Downtime: {downtime_pct:.2f}%")
print(f"Average Idle: {idle_pct:.2f}%")
print(f"Average Performance Loss: {performance_loss_pct:.2f}%")
print(f"Average Health Score: {avg_health:.2f}")

Average Production: 63.25%
Average Downtime: 15.95%
Average Idle: 3.57%
Average Performance Loss: 11.16%
Average Health Score: 99.69


In [30]:
anomaly_data = evidence["anomaly_data"]

if len(anomaly_data) == 0:
    anomaly_count = 0
    anomaly_status = "NO DATA AVAILABLE"

elif "is_anomaly" in anomaly_data.columns:

    anomaly_count = int(
        anomaly_data["is_anomaly"].sum()
    )

    if anomaly_count > 0:
        anomaly_status = "ANOMALY DETECTED"
    else:
        anomaly_status = "NO ANOMALY DETECTED"

else:
    anomaly_count = 0
    anomaly_status = "ANOMALY DATA UNAVAILABLE"

print("Anomalous Records:", anomaly_count)
print("Anomaly Status:", anomaly_status)

Anomalous Records: 421
Anomaly Status: ANOMALY DETECTED


In [31]:
risk_data = evidence["risk_data"]

if len(risk_data) == 0:

    risk_count = 0
    risk_status = "NO DATA AVAILABLE"
    latest_risk_level = "UNKNOWN"

elif "risk_prediction" in risk_data.columns:

    risk_count = int(
        risk_data["risk_prediction"].sum()
    )

    if risk_count > 0:
        risk_status = "RISK DETECTED"
    else:
        risk_status = "NO HIGH-RISK PREDICTION"

    if "risk_level" in risk_data.columns:
        latest_risk_level = str(
            risk_data["risk_level"].iloc[-1]
        )
    else:
        latest_risk_level = "UNKNOWN"

else:

    risk_count = 0
    risk_status = "RISK DATA UNAVAILABLE"
    latest_risk_level = "UNKNOWN"


print("High-Risk Predictions:", risk_count)
print("Risk Status:", risk_status)
print("Latest Risk Level:", latest_risk_level)

High-Risk Predictions: 378
Risk Status: RISK DETECTED
Latest Risk Level: LOW


In [32]:
machine_text = machine_data.tail(10).to_string(
    index=False
)

anomaly_text = anomaly_data.tail(10).to_string(
    index=False
)

risk_text = risk_data.tail(10).to_string(
    index=False
)

print("Evidence prepared successfully")

Evidence prepared successfully


In [35]:
import chromadb
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully


In [36]:
chroma_client = chromadb.PersistentClient(
    path=r"C:\Packaging_AI_Control_Tower\data\chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="packaging_knowledge"
)

print("RAG knowledge base connected")
print("Documents in collection:", collection.count())

RAG knowledge base connected
Documents in collection: 21


In [37]:
def search_knowledge(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query]
    )[0]

    results = collection.query(
        query_embeddings=[
            query_embedding.tolist()
        ],
        n_results=top_k
    )

    return results

In [38]:
def get_knowledge(query, top_k=5):

    results = search_knowledge(
        query,
        top_k=top_k
    )

    context_parts = []

    for i, document in enumerate(
        results["documents"][0]
    ):

        source = results["metadatas"][0][i]["source"]

        context_parts.append(
            f"""
SOURCE: {source}

CONTENT:
{document}
"""
        )

    return "\n\n".join(context_parts)

In [39]:
question = """
Machine has high downtime and repeated alarms.
What should be investigated?
"""

knowledge = get_knowledge(
    question,
    top_k=5
)

print(knowledge)


SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
c component has failed.

2. ALARM INVESTIGATION

When an alarm is detected, the control tower should examine:

- Equipment ID
- Alarm identifier
- Alarm occurrence time
- Alarm duration when available
- Machine production around the event
- Downtime around the event
- Performance loss around the event
- Previous alarms
- Anomaly detection results
- Risk prediction results

3. RECENT ALARM ANALYSIS

The system should determine whether the alarm is:

- A single isolated event
- Repeated over a short period
- Associated with production reduction
- Associated with increased downtime
- Associated with an anomaly
- Associated with elevated predicted risk

Repeated events may require additional investigation.

4. ALARM FREQUENCY

A high alarm count may indicate repeated operational events.

Howev



SOURCE: ..\documents\alarm_guides\packaging_alarm_investigation.txt

CONTENT:
estigation.

4. ALARM FREQUENCY

A high

In [40]:
question = f"""
Machine {equipment_id} has operational issues.
Analyze downtime, production, performance loss,
alarms, anomaly status and operational risk.
What possible causes should be investigated?
"""

knowledge = get_knowledge(
    question,
    top_k=5
)

print(knowledge)


SOURCE: ..\documents\maintenance_guides\packaging_maintenance_guidelines.txt

CONTENT:
activity
- Production reduction
- Repeated operational interruptions
- Recent machine changes
- Historical downtime pattern

3. LOW PRODUCTION

When production decreases, investigate:

- Downtime
- Idle time
- Performance loss
- Operational changes
- Alarm events
- Recent machine history

Low production does not automatically mean equipment failure.

4. PERFORMANCE LOSS

When performance loss increases, compare it with:

- Production
- Downtime
- Idle time
- Historical performance
- Anomaly score
- Predicted risk

Multiple indicators showing deterioration provide stronger evidence than one indicator alone.

5. REPEATED EVENTS

Repeated abnormal events should be investigated.

The system can identify:

- Repeated anomalies
- Repeated alarms
- Repeated downtime
- Repeated production deterio



SOURCE: ..\documents\machine_manuals\packaging_machine_operations.txt

CONTENT:
utput and may indicate an ope

In [41]:
rca_prompt = f"""
You are an AI Root Cause Analysis Assistant
for a Packaging Industry Operations Control Tower.

MACHINE:
{equipment_id}

OPERATIONAL EVIDENCE:
{machine_text}

ANOMALY DETECTION RESULTS:
{anomaly_text}

RISK PREDICTION RESULTS:
{risk_text}

SUMMARY METRICS:

Average Production:
{production_pct:.2f}%

Average Downtime:
{downtime_pct:.2f}%

Average Idle:
{idle_pct:.2f}%

Average Performance Loss:
{performance_loss_pct:.2f}%

Average Health Score:
{avg_health:.2f}

ANOMALY STATUS:
{anomaly_status}

ANOMALOUS RECORD COUNT:
{anomaly_count}

RISK STATUS:
{risk_status}

HIGH-RISK PREDICTION COUNT:
{risk_count}

LATEST RISK LEVEL:
{latest_risk_level}

RETRIEVED KNOWLEDGE:
{knowledge}

IMPORTANT RULES:

1. Separate observed evidence from possible causes.

2. Do not invent the meaning of any alarm code.

3. Do not claim a possible cause is confirmed.

4. Use only the available evidence and retrieved knowledge.

5. If evidence is insufficient, clearly say so.

6. Possible root causes must be treated as hypotheses.

7. Do not invent sensor readings or machine conditions.

8. Physical machine intervention requires human verification.

Return the result in this exact structure:

Machine:

Incident Summary:

Observed Evidence:

Anomaly Status:

Risk Status:

Possible Root Cause:

Supporting Evidence:

Confidence:

Recommended Investigation:

Recommended Action:

Human Verification:
"""

In [42]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=rca_prompt
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Machine:
s_1

Incident Summary:
The machine s_1 is exhibiting fluctuating operational performance characterized by high variance in production rates and recurring cycles of downtime. While the most recent risk prediction indicates a "LOW" risk level, the system has flagged an anomalous state across multiple intervals, suggesting instability in the packaging process.

Observed Evidence:
- **Production Volatility:** Production percentages show significant swings, ranging from ~22% to ~88% in recent intervals.
- **Downtime Patterns:** Downtime percentages are notably high, peaking at ~52.38% (08:00:00) and ~50.36% (06:00:00).
- **Operational Transitions:** Frequent changes are recorded, with peaks of 21 changes observed at 06:00:00 and 19 changes at 13:00:00.
- **Health Score:** The health score remains high (near 99.3 - 99.9 range) despite the observed fluctuations in production and downtime.
- **Anomaly Detection:** Anomaly prediction flags have been triggered, though the system classif

In [43]:
rca_result = response.text

rca_record = pd.DataFrame([
    {
        "equipment_ID": equipment_id,
        "anomaly_status": anomaly_status,
        "anomaly_count": anomaly_count,
        "risk_status": risk_status,
        "risk_count": risk_count,
        "risk_level": latest_risk_level,
        "average_production": avg_production,
        "average_downtime": avg_downtime,
        "average_idle": avg_idle,
        "average_performance_loss": avg_performance_loss,
        "average_health_score": avg_health,
        "rca_result": rca_result
    }
])

print(rca_record)

  equipment_ID    anomaly_status  anomaly_count    risk_status  risk_count  \
0          s_1  ANOMALY DETECTED            421  RISK DETECTED         378   

  risk_level  average_production  average_downtime  average_idle  \
0        LOW            0.632548          0.159515       0.03571   

   average_performance_loss  average_health_score  \
0                  0.111561             99.693214   

                                          rca_result  
0  Machine:\ns_1\n\nIncident Summary:\nThe machin...  


In [44]:
output_path = r"C:\Packaging_AI_Control_Tower\Data\processed\rca_results.csv"

rca_record.to_csv(
    output_path,
    index=False
)

print("RCA result saved successfully")
print(output_path)

RCA result saved successfully
C:\Packaging_AI_Control_Tower\Data\processed\rca_results.csv
